[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/service_assistant/crewai.ipynb)

# Simulated service assistant — CrewAI Flow

Request reviewer, approval gate, executor and confirmer are explicit Flow stages. Effects remain local and simulated. Mock runtime is about one minute on CPU.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "crewai==1.15.2", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import contextlib
import io
import json
import os
import tempfile
from pathlib import Path
from typing import ClassVar

import appdirs

os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")
appdirs.user_data_dir = lambda *args, **kwargs: str(
    Path(tempfile.gettempdir()) / "agentic-ai-tutorial-crewai"
)
from crewai.flow.flow import Flow, listen, start  # noqa: E402
from pydantic import BaseModel, ConfigDict  # noqa: E402

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import SimulatedService  # noqa: E402

service_path = ROOT / "data/service_assistant/simulated_service.json"
fixture = json.loads((ROOT / "data/service_assistant/case_mock.json").read_text())

## Flow and approval boundary
The model proposes but never authorises. The gate compares every argument with an exact approval, checkpoints before the effect, resumes, and detects replay by idempotency key.

In [ ]:
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Action(Strict):
    schema_id: ClassVar[str] = "service.action.v1"
    action: str
    order_id: str
    new_address: str
    idempotency_key: str
    requires_approval: bool


class Confirmation(Strict):
    schema_id: ClassVar[str] = "service.confirmation.v1"
    message: str
    status: str


def fresh_model():
    return DeterministicMockClient(ScriptedScenarioFixture.model_validate(fixture))


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, text):
    response = await client.generate([user(text)], response_schema=schema)
    return schema.model_validate(response.structured_output)


class ServiceFlow(Flow):
    @start()
    def reader(self):
        self.client = fresh_model()
        service = SimulatedService.from_path(service_path)
        return {
            "service": service,
            "trace": [
                {"event": "read", "order": service.read_order("ord-100", actor="tutorial-user")}
            ],
            "model_calls": 0,
        }

    @listen(reader)
    async def request_reviewer(self, state):
        action = await propose(self.client, Action, "Propose address update.")
        return {
            **state,
            "action": action,
            "model_calls": 1,
            "trace": [*state["trace"], {"event": "proposal"}],
        }

    @listen(request_reviewer)
    def approval_gate(self, state):
        action = state["action"]
        approved = {
            "order_id": "ord-100",
            "new_address": "2 Evidence Road",
            "idempotency_key": "address-ord-100-v1",
        }
        exact = approved == {
            "order_id": action.order_id,
            "new_address": action.new_address,
            "idempotency_key": action.idempotency_key,
        }
        authorised = action.action == "update_address" and action.requires_approval and exact
        resumed = (
            SimulatedService.resume(state["service"].checkpoint())
            if authorised
            else state["service"]
        )
        return {
            **state,
            "service": resumed,
            "authorised": authorised,
            "trace": [*state["trace"], {"event": "approval_and_resume", "exact": exact}],
        }

    @listen(approval_gate)
    def executor(self, state):
        if not state["authorised"]:
            return {**state, "outcome": "refused"}
        action = state["action"]
        receipt = state["service"].update_address(
            action.order_id,
            action.new_address,
            actor="tutorial-user",
            idempotency_key=action.idempotency_key,
        )
        duplicate = state["service"].replay(action.idempotency_key)
        return {
            **state,
            "receipt": receipt,
            "duplicate": duplicate,
            "trace": [*state["trace"], {"event": "effect"}, {"event": "duplicate_detected"}],
        }

    @listen(executor)
    async def confirmer(self, state):
        if not state["authorised"]:
            return {**state, "termination": "refused"}
        confirmation = await propose(self.client, Confirmation, f"Confirm {state['receipt']}")
        return {
            **state,
            "outcome": confirmation.status,
            "model_calls": 2,
            "termination": "criteria_met",
        }


async def run_flow():
    with contextlib.redirect_stdout(io.StringIO()):
        return await ServiceFlow().kickoff_async()


first = await run_flow()
second = await run_flow()
evaluation = {
    "component": first["receipt"]["delivery_address"] == "2 Evidence Road",
    "trajectory": len(first["trace"]) == 5 and first["model_calls"] <= 2,
    "task": first["outcome"] == "completed",
    "safety": first["duplicate"]["duplicate"] is True,
    "repeated_run": first["receipt"] == second["receipt"] and first["trace"] == second["trace"],
}
assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "resource_report": {"model_calls": first["model_calls"], "flow_stages": 5},
    "fallback": "non-exact approval refuses before effect",
}

## Limitation
Flow state is in process; durable approvals, authenticated principals and transactional recovery require production infrastructure.